# Normalisation

## Overview
**Normalisation** is the process of organising a database schema to reduce data redundancy and avoid update anomalies. Each normal form (NF) adds a stricter constraint on top of the previous one.

**The three main anomalies that normalisation prevents:**

| Anomaly | Problem | Example |
|---|---|---|
| **Insert anomaly** | Can't add data without adding unrelated data | Can't add a new department until a provider joins it |
| **Update anomaly** | Changing one fact requires updating many rows | Updating a provider's department name in 50 encounter rows |
| **Delete anomaly** | Deleting one fact deletes unrelated data | Deleting the last encounter for a department removes all dept info |

**Normal forms:**

| NF | Rule | Eliminates |
|---|---|---|
| **1NF** | Atomic values; no repeating groups; each row uniquely identifiable | Multi-valued cells, repeating columns |
| **2NF** | 1NF + every non-key attribute fully depends on the whole primary key | Partial dependencies (composite-PK tables) |
| **3NF** | 2NF + no transitive dependencies (non-key -> non-key) | Transitive dependencies |
| **BCNF** | Every determinant is a candidate key | Edge cases 3NF misses with overlapping candidate keys |

**Practical guidance:** Most OLTP schemas target 3NF. Analytical/warehouse schemas are intentionally denormalised for query performance (see `20_data_warehousing_patterns`).

---

In [1]:
import sqlite3
import pandas as pd
conn = sqlite3.connect(":memory:")
conn.executescript("""CREATE TABLE patients (patient_id INTEGER PRIMARY KEY, first_name TEXT NOT NULL, last_name TEXT NOT NULL, dob TEXT NOT NULL, sex TEXT CHECK(sex IN ('M','F','O')), province TEXT, active INTEGER DEFAULT 1);CREATE TABLE departments (dept_id INTEGER PRIMARY KEY, dept_name TEXT NOT NULL UNIQUE, building TEXT, floor_no INTEGER);CREATE TABLE providers (provider_id INTEGER PRIMARY KEY, full_name TEXT NOT NULL, specialty TEXT, licence_no TEXT UNIQUE, province TEXT, dept_id INTEGER REFERENCES departments(dept_id), manager_id INTEGER REFERENCES providers(provider_id));CREATE TABLE diagnoses (diag_code TEXT PRIMARY KEY, description TEXT NOT NULL, category TEXT NOT NULL);CREATE TABLE encounters (enc_id INTEGER PRIMARY KEY, patient_id INTEGER NOT NULL REFERENCES patients(patient_id), provider_id INTEGER NOT NULL REFERENCES providers(provider_id), dept_id INTEGER REFERENCES departments(dept_id), enc_date TEXT NOT NULL, cost_cad REAL, admitted INTEGER DEFAULT 0 CHECK(admitted IN (0,1)));CREATE TABLE encounter_diagnoses (enc_id INTEGER NOT NULL REFERENCES encounters(enc_id), diag_code TEXT NOT NULL REFERENCES diagnoses(diag_code), diag_rank INTEGER DEFAULT 1, confirmed INTEGER DEFAULT 1, PRIMARY KEY (enc_id, diag_code));CREATE TABLE lab_results (result_id INTEGER PRIMARY KEY, patient_id INTEGER NOT NULL REFERENCES patients(patient_id), test_name TEXT NOT NULL, result_val REAL, units TEXT, collected TEXT NOT NULL, flagged INTEGER DEFAULT 0);INSERT INTO departments VALUES (1,'Emergency','Tower A',1),(2,'Cardiology','Tower B',2),(3,'Oncology','Tower C',3),(4,'Family Medicine','Clinic',1),(5,'Orthopaedics','Tower A',2),(6,'Radiology','Tower B',3),(7,'ICU','Tower C',NULL);INSERT INTO providers VALUES (10,'Dr. Sarah MacNeil','Cardiology','MC-1001','NB',2,NULL),(11,'Dr. James Wong','Oncology','MC-1002','BC',3,10),(12,'Dr. Fatima Osei','Family Medicine','MC-1003','ON',4,10),(13,'Dr. Ethan Larson','Orthopaedics','MC-1004','QC',5,10),(14,'Dr. Priya Sharma','Emergency','MC-1005','AB',1,10),(15,'Dr. Lucas Petit','Radiology','MC-1006','QC',6,11);INSERT INTO patients VALUES (1,'Aroha','Ngata','1985-03-12','F','NB',1),(2,'Liam','Chen','1972-11-04','M','NS',1),(3,'Fatima','Al-Rashid','1990-07-22','F','ON',1),(4,'James','MacLeod','1955-01-30','M','NB',0),(5,'Sofia','Petrov','2001-09-15','F','BC',1),(6,'Noah','Williams','1968-06-08','M','AB',1),(7,'Mei','Zhang','1995-04-17','F','ON',1),(8,'Ethan','Tremblay','1980-12-01','M','QC',0),(9,'Priya','Nair','1977-08-25','F','BC',1),(10,'Marcus','Okafor','1993-05-19','M','ON',1),(11,'Diana','Flores','2000-02-14','F','NB',1);INSERT INTO diagnoses VALUES ('I25.1','Atherosclerotic heart disease','Cardiovascular'),('J18.9','Pneumonia, unspecified','Respiratory'),('Z00.0','General medical examination','Preventive'),('M17.1','Primary osteoarthritis of knee','Musculoskeletal'),('J06.9','Acute upper respiratory infection','Respiratory'),('R07.9','Chest pain, unspecified','Symptoms'),('I10','Essential hypertension','Cardiovascular'),('R55','Syncope and collapse','Symptoms'),('I48.0','Paroxysmal atrial fibrillation','Cardiovascular'),('S52.5','Fracture of lower end of radius','Injury'),('E11.9','Type 2 diabetes mellitus','Endocrine'),('I50.9','Heart failure, unspecified','Cardiovascular');INSERT INTO encounters VALUES (1,1,10,2,'2023-01-15',3200.00,1),(2,2,14,1,'2023-02-20',1850.00,1),(3,3,12,4,'2023-03-05',120.00,0),(4,4,13,5,'2023-03-18',5500.00,1),(5,5,12,4,'2023-04-02',95.00,0),(6,6,14,1,'2023-05-11',780.00,0),(7,7,10,2,'2023-06-22',2100.00,1),(8,8,12,4,'2023-07-14',80.00,0),(9,1,14,1,'2023-08-30',410.00,0),(10,3,12,4,'2023-09-12',110.00,0),(11,9,10,2,'2023-10-01',1750.00,1),(12,10,14,1,'2023-11-03',2200.00,1),(13,2,10,2,'2023-11-20',2900.00,1),(14,6,12,4,'2023-12-01',115.00,0);INSERT INTO encounter_diagnoses VALUES (1,'I25.1',1,1),(1,'I10',2,1),(2,'J18.9',1,1),(3,'Z00.0',1,1),(4,'M17.1',1,1),(5,'J06.9',1,1),(6,'R07.9',1,1),(7,'I10',1,1),(7,'I48.0',2,1),(9,'R55',1,1),(10,'Z00.0',1,1),(11,'I48.0',1,1),(12,'S52.5',1,1),(13,'I25.1',1,1),(14,'Z00.0',1,1);INSERT INTO lab_results VALUES (1,1,'HbA1c',7.2,'%','2023-03-10',0),(2,2,'HbA1c',9.1,'%','2023-03-15',1),(3,3,'Creatinine',88.5,'umol/L','2023-04-01',0),(4,4,'Creatinine',145.0,'umol/L','2023-04-12',1),(5,5,'HbA1c',5.8,'%','2023-05-05',0),(6,6,'Sodium',138.0,'mmol/L','2023-05-20',0),(7,7,'Sodium',151.0,'mmol/L','2023-06-01',1),(8,1,'Creatinine',72.3,'umol/L','2023-06-18',0),(9,8,'HbA1c',6.4,'%','2023-07-02',0),(10,9,'Creatinine',210.0,'umol/L','2023-07-15',1),(11,2,'Creatinine',95.0,'umol/L','2023-08-01',0),(12,10,'HbA1c',7.8,'%','2023-08-20',1);""")
conn.commit()
print("Healthcare schema ready.")
for t in ["patients","providers","departments","diagnoses","encounters","encounter_diagnoses","lab_results"]:
    n = conn.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
    print(f"  {t}: {n} rows")

Healthcare schema ready.
  patients: 11 rows
  providers: 6 rows
  departments: 7 rows
  diagnoses: 12 rows
  encounters: 14 rows
  encounter_diagnoses: 15 rows
  lab_results: 12 rows


---
## 1NF -- atomic values and no repeating groups

In [2]:
# Show a denormalised table violating 1NF, then fix it
print("=== 1NF VIOLATION: multi-valued cell ===")
conn.execute("""CREATE TEMP TABLE enc_denorm_1nf (
    enc_id INTEGER, patient_name TEXT, diag_codes TEXT, cost_cad REAL)""")
conn.executemany("INSERT INTO enc_denorm_1nf VALUES (?,?,?,?)", [
    (1, "Aroha Ngata",   "I25.1, I10",       3200.00),
    (2, "Liam Chen",     "J18.9",             1850.00),
    (7, "Mei Zhang",     "I10, I48.0, E11.9", 2100.00),
])
conn.commit()
print("Violates 1NF -- diag_codes contains multiple values in one cell:")
print(pd.read_sql("SELECT * FROM enc_denorm_1nf", conn).to_string(index=False))

print()
print("=== 1NF FIX: one diagnosis per row (use bridge table) ===")
q = """
SELECT e.enc_id, p.first_name || ' ' || p.last_name AS patient_name,
       ed.diag_code, ed.diag_rank, e.cost_cad
FROM   encounters AS e
JOIN   patients AS p ON e.patient_id = p.patient_id
JOIN   encounter_diagnoses AS ed ON e.enc_id = ed.enc_id
WHERE  e.enc_id IN (1,2,7)
ORDER BY e.enc_id, ed.diag_rank
"""
print("1NF-compliant: one value per cell, separate row per diagnosis:")
print(pd.read_sql(q, conn).to_string(index=False))


=== 1NF VIOLATION: multi-valued cell ===
Violates 1NF -- diag_codes contains multiple values in one cell:
 enc_id patient_name        diag_codes  cost_cad
      1  Aroha Ngata        I25.1, I10    3200.0
      2    Liam Chen             J18.9    1850.0
      7    Mei Zhang I10, I48.0, E11.9    2100.0

=== 1NF FIX: one diagnosis per row (use bridge table) ===
1NF-compliant: one value per cell, separate row per diagnosis:
 enc_id patient_name diag_code  diag_rank  cost_cad
      1  Aroha Ngata     I25.1          1    3200.0
      1  Aroha Ngata       I10          2    3200.0
      2    Liam Chen     J18.9          1    1850.0
      7    Mei Zhang       I10          1    2100.0
      7    Mei Zhang     I48.0          2    2100.0


---
## 2NF -- eliminating partial dependencies

In [3]:
# 2NF violation: composite PK, and a non-key column depends on only part of the PK
print("=== 2NF VIOLATION: partial dependency on composite PK ===")
conn.execute("""CREATE TEMP TABLE enc_prov_denorm (
    enc_id INTEGER, provider_id INTEGER, enc_date TEXT,
    provider_name TEXT,  -- depends only on provider_id, not enc_id
    specialty TEXT,      -- depends only on provider_id, not enc_id
    cost_cad REAL,
    PRIMARY KEY (enc_id, provider_id))""")
conn.executemany("INSERT INTO enc_prov_denorm VALUES (?,?,?,?,?,?)", [
    (1, 10, "2023-01-15", "Dr. Sarah MacNeil", "Cardiology",     3200.00),
    (7, 10, "2023-06-22", "Dr. Sarah MacNeil", "Cardiology",     2100.00),
    (13,10, "2023-11-20", "Dr. Sarah MacNeil", "Cardiology",     2900.00),
    (2, 14, "2023-02-20", "Dr. Priya Sharma",  "Emergency",      1850.00),
    (6, 14, "2023-05-11", "Dr. Priya Sharma",  "Emergency",       780.00),
])
conn.commit()
print("provider_name and specialty depend only on provider_id -- not on the full PK (enc_id, provider_id):")
print(pd.read_sql("SELECT * FROM enc_prov_denorm", conn).to_string(index=False))
print()
print("Update anomaly: changing 'Dr. Sarah MacNeil' -> 'Dr. S. MacNeil' requires 3 row updates.")

print()
print("=== 2NF FIX: move provider attributes to the providers table ===")
print("encounters table: (enc_id PK, patient_id FK, provider_id FK, enc_date, cost_cad)")
print("providers table:  (provider_id PK, full_name, specialty, ...)")
print("No partial dependencies -- each non-key column depends on the full PK of its table.")


=== 2NF VIOLATION: partial dependency on composite PK ===
provider_name and specialty depend only on provider_id -- not on the full PK (enc_id, provider_id):
 enc_id  provider_id   enc_date     provider_name  specialty  cost_cad
      1           10 2023-01-15 Dr. Sarah MacNeil Cardiology    3200.0
      7           10 2023-06-22 Dr. Sarah MacNeil Cardiology    2100.0
     13           10 2023-11-20 Dr. Sarah MacNeil Cardiology    2900.0
      2           14 2023-02-20  Dr. Priya Sharma  Emergency    1850.0
      6           14 2023-05-11  Dr. Priya Sharma  Emergency     780.0

Update anomaly: changing 'Dr. Sarah MacNeil' -> 'Dr. S. MacNeil' requires 3 row updates.

=== 2NF FIX: move provider attributes to the providers table ===
encounters table: (enc_id PK, patient_id FK, provider_id FK, enc_date, cost_cad)
providers table:  (provider_id PK, full_name, specialty, ...)
No partial dependencies -- each non-key column depends on the full PK of its table.


---
## 3NF -- eliminating transitive dependencies

In [4]:
# 3NF violation: non-key column determines another non-key column
print("=== 3NF VIOLATION: transitive dependency ===")
conn.execute("""CREATE TEMP TABLE prov_denorm_3nf (
    provider_id INTEGER PRIMARY KEY,
    full_name TEXT,
    dept_id INTEGER,
    dept_name TEXT,   -- depends on dept_id, not provider_id directly
    building TEXT)    -- depends on dept_id, not provider_id directly
""")
conn.executemany("INSERT INTO prov_denorm_3nf VALUES (?,?,?,?,?)", [
    (10,"Dr. Sarah MacNeil",  2,"Cardiology",     "Tower B"),
    (11,"Dr. James Wong",     3,"Oncology",        "Tower C"),
    (12,"Dr. Fatima Osei",    4,"Family Medicine", "Clinic"),
    (14,"Dr. Priya Sharma",   1,"Emergency",       "Tower A"),
])
conn.commit()
print("dept_name and building depend on dept_id, not on provider_id directly:")
print("  provider_id -> dept_id -> dept_name (transitive dependency)")
print(pd.read_sql("SELECT * FROM prov_denorm_3nf", conn).to_string(index=False))
print()
print("Update anomaly: renaming 'Tower B' to 'North Tower' requires updating multiple rows.")

print()
print("=== 3NF FIX: move dept attributes to departments table ===")
print("providers:    (provider_id PK, full_name, specialty, dept_id FK)")
print("departments:  (dept_id PK, dept_name, building)")
print()
print("Verified in our normalised schema:")
q = """
SELECT p.provider_id, p.full_name, p.dept_id, d.dept_name, d.building
FROM   providers AS p
JOIN   departments AS d ON p.dept_id = d.dept_id
ORDER BY p.provider_id
"""
print(pd.read_sql(q, conn).to_string(index=False))
conn.close()


=== 3NF VIOLATION: transitive dependency ===
dept_name and building depend on dept_id, not on provider_id directly:
  provider_id -> dept_id -> dept_name (transitive dependency)
 provider_id         full_name  dept_id       dept_name building
          10 Dr. Sarah MacNeil        2      Cardiology  Tower B
          11    Dr. James Wong        3        Oncology  Tower C
          12   Dr. Fatima Osei        4 Family Medicine   Clinic
          14  Dr. Priya Sharma        1       Emergency  Tower A

Update anomaly: renaming 'Tower B' to 'North Tower' requires updating multiple rows.

=== 3NF FIX: move dept attributes to departments table ===
providers:    (provider_id PK, full_name, specialty, dept_id FK)
departments:  (dept_id PK, dept_name, building)

Verified in our normalised schema:
 provider_id         full_name  dept_id       dept_name building
          10 Dr. Sarah MacNeil        2      Cardiology  Tower B
          11    Dr. James Wong        3        Oncology  Tower C
       

---
## Common Pitfalls

**1. Storing lists in a single column -- the most common 1NF violation**
`diag_codes = 'I25.1, I10, E11.9'` in one cell means you cannot query individual diagnoses without string parsing, cannot join to a diagnoses reference table, and cannot enforce referential integrity. Use a bridge table.

**2. Over-normalising reporting queries**
3NF is ideal for OLTP (transactional) schemas. For analytical queries that join many tables, the cost of multiple joins can be high. Data warehouses intentionally denormalise into star schemas (see folder 20). Know when to denormalise.

**3. Not recognising partial dependencies in composite-PK tables**
The 2NF rule only applies to tables with composite PKs. Single-column PK tables skip 2NF. In tables like `encounter_diagnoses (enc_id, diag_code)`, any attribute that depends only on `enc_id` OR only on `diag_code` violates 2NF and should move to the encounters or diagnoses table.

**4. Confusing transitive dependencies with legitimate functional dependencies**
`provider_id -> dept_id` is a legitimate dependency (a provider belongs to a department). `dept_id -> dept_name` is also legitimate. The 3NF violation is the chain: `provider_id -> dept_id -> dept_name`. Break it by putting `dept_name` on the `departments` table, not the `providers` table.

**5. Normalising lookup/reference data differently from transactional data**
Reference tables like `diagnoses (diag_code, description, category)` are already in 3NF because each attribute depends only on `diag_code`. The effort of normalisation applies mainly to transactional tables with many repeated facts.

**6. Treating normalisation as a one-time exercise**
Schemas denormalise over time as features are added. `ALTER TABLE providers ADD COLUMN dept_name TEXT` is a 3NF violation added in a hurry. Schedule periodic schema reviews to catch drift.


---
*sql_methods_library - Samantha McGarrigle*